# soxlbacktest_reference_model_score_v1

Purpose: use the qlib-aligned SOXLBacktest reference from `soxlbacktest_qlib_walkforward_baseline_v1` to score current replayable/trained model candidates.

Governance:
- Canonical reference baseline: `soxl_backtest_policy_reimplementation_v1` / `legacy_default` qlib-aligned daily path.
- Direct comparison baseline: `legacy_default` for buy/sell/position timing and return floor; `wf_pass_ytd18_15_15_02` as a low-drawdown reference floor.
- Role: `diagnose` and next-training source selection. This card does not promote a model.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`, and `ytd_2026` through the reference common end.
- Changed layer: `selection` only; reusable model code is not modified in this notebook.
- Hypothesis: no existing replayable actor currently satisfies both full walk-forward drawdown and return-alignment gates, but the SOXLBacktest reference can identify the closest source family for the next training loop.


In [1]:

from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise FileNotFoundError(f"Could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD = ROOT / "runtime" / "model_selection" / "card" / "soxlbacktest_reference_model_score_v1"
CARD.mkdir(parents=True, exist_ok=True)
ARTIFACTS = {
    "candidate_score_summary": CARD / "candidate_score_summary.csv",
    "fold_alignment_metrics": CARD / "fold_alignment_metrics.csv",
    "profile_source_audit": CARD / "profile_source_audit.csv",
    "missing_evidence": CARD / "missing_evidence.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "conclusion": CARD / "conclusion.md",
}


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


print({"card": rel(CARD), "artifact_count": len(ARTIFACTS)})


{'card': 'runtime/model_selection/card/soxlbacktest_reference_model_score_v1', 'artifact_count': 7}


In [2]:

HANDOFF = json.loads((ROOT / "runtime/model_selection/card/retraining_gate_gap_handoff_v1/training_objective_handoff.json").read_text())
BASELINE_CARD = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1"
BASELINE_SUMMARY = json.loads((BASELINE_CARD / "summary.json").read_text())
REFERENCE_DAILY = pd.read_csv(BASELINE_CARD / "candidate_daily.csv")
REFERENCE_METRICS = pd.read_csv(BASELINE_CARD / "candidate_metrics.csv")
REFERENCE_GATE = pd.read_csv(BASELINE_CARD / "candidate_gate_summary.csv")

REQUIRED_WF = list(HANDOFF["retention_gate"]["required_wf_windows"])
MAX_WF_DD = float(HANDOFF["retention_gate"]["max_walkforward_drawdown_pct"])
REQ_SOXL_RETURN = float(HANDOFF["required_soxlbacktest_return_pct"])
REQ_BEST_RETURN = float(HANDOFF["required_best_return_pct"])
YTD_START = str(REFERENCE_METRICS[(REFERENCE_METRICS["candidate"].eq("legacy_default")) & (REFERENCE_METRICS["window"].eq("ytd_2026"))].iloc[0]["start"])
YTD_END = str(REFERENCE_METRICS[(REFERENCE_METRICS["candidate"].eq("legacy_default")) & (REFERENCE_METRICS["window"].eq("ytd_2026"))].iloc[0]["end"])

thresholds = {
    "required_wf_windows": REQUIRED_WF,
    "max_walkforward_drawdown_pct": MAX_WF_DD,
    "required_soxlbacktest_return_pct": REQ_SOXL_RETURN,
    "required_best_return_pct": REQ_BEST_RETURN,
    "ytd_start": YTD_START,
    "ytd_end": YTD_END,
}
thresholds


{'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'max_walkforward_drawdown_pct': 20.0,
 'required_soxlbacktest_return_pct': 132.077978,
 'required_best_return_pct': 213.143462,
 'ytd_start': '2026-01-02',
 'ytd_end': '2026-06-09'}

In [3]:

def normalise_daily(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    if "date" not in df.columns and "datetime" in df.columns:
        df = df.rename(columns={"datetime": "date"})
    if "date" not in df.columns:
        raise ValueError("daily frame missing date/datetime column")
    if "close" not in df.columns and "price" in df.columns:
        df["close"] = df["price"]
    if "price" not in df.columns and "close" in df.columns:
        df["price"] = df["close"]
    if "action" not in df.columns:
        df["action"] = "HOLD"
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    for col in ["strategy_nav", "price", "close", "position"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["action"] = df["action"].fillna("HOLD").astype(str).str.upper()
    return df.sort_values("date").reset_index(drop=True)


def window_slice(df: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    daily = normalise_daily(df)
    return daily[(daily["date"] >= start) & (daily["date"] <= end)].copy()


def max_drawdown_pct(nav: pd.Series) -> float:
    values = pd.to_numeric(nav, errors="coerce").dropna().to_numpy(dtype=float)
    if len(values) == 0:
        return math.nan
    peaks = np.maximum.accumulate(values)
    return float(np.max((peaks - values) / peaks) * 100.0)


def return_pct(nav: pd.Series) -> float:
    values = pd.to_numeric(nav, errors="coerce").dropna().to_numpy(dtype=float)
    if len(values) < 2 or values[0] == 0:
        return math.nan
    return float((values[-1] / values[0] - 1.0) * 100.0)


def action_event_mask(actions: pd.Series) -> pd.Series:
    upper = actions.fillna("HOLD").astype(str).str.upper()
    return ~upper.isin({"HOLD", "", "NAN", "NONE"})


def align_to_reference(candidate: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    left = normalise_daily(candidate).set_index("date")
    right = normalise_daily(reference).set_index("date")
    common = left.index.intersection(right.index).sort_values()
    if len(common) == 0:
        return pd.DataFrame()
    merged = left.loc[common].join(right.loc[common], how="inner", lsuffix="_candidate", rsuffix="_reference")
    merged = merged.reset_index().rename(columns={"index": "date"})
    return merged


def alignment_metrics(candidate: pd.DataFrame, reference: pd.DataFrame) -> dict[str, Any]:
    merged = align_to_reference(candidate, reference)
    if merged.empty:
        return {
            "aligned_rows": 0,
            "date_overlap_start": "",
            "date_overlap_end": "",
            "position_mae": math.nan,
            "position_corr": math.nan,
            "event_match_rate": math.nan,
            "candidate_event_days": 0,
            "reference_event_days": 0,
            "action_mismatch_days": 0,
        }
    cand_pos = pd.to_numeric(merged.get("position_candidate"), errors="coerce").fillna(0.0)
    ref_pos = pd.to_numeric(merged.get("position_reference"), errors="coerce").fillna(0.0)
    cand_event = action_event_mask(merged.get("action_candidate", pd.Series(index=merged.index, dtype=object)))
    ref_event = action_event_mask(merged.get("action_reference", pd.Series(index=merged.index, dtype=object)))
    corr = cand_pos.corr(ref_pos) if cand_pos.std() > 0 and ref_pos.std() > 0 else math.nan
    return {
        "aligned_rows": int(len(merged)),
        "date_overlap_start": str(merged["date"].min()),
        "date_overlap_end": str(merged["date"].max()),
        "position_mae": float((cand_pos - ref_pos).abs().mean()),
        "position_corr": float(corr) if pd.notna(corr) else math.nan,
        "event_match_rate": float((cand_event == ref_event).mean()),
        "candidate_event_days": int(cand_event.sum()),
        "reference_event_days": int(ref_event.sum()),
        "action_mismatch_days": int((cand_event != ref_event).sum()),
    }


def metric_from_daily(daily: pd.DataFrame) -> dict[str, Any]:
    if daily.empty or "strategy_nav" not in daily.columns:
        return {"return_pct": math.nan, "max_drawdown_pct": math.nan, "rows": int(len(daily))}
    return {
        "return_pct": return_pct(daily["strategy_nav"]),
        "max_drawdown_pct": max_drawdown_pct(daily["strategy_nav"]),
        "rows": int(len(daily)),
    }


In [4]:

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def metric_from_summary(path: Path, window: str) -> dict[str, Any] | None:
    try:
        payload = load_json(path)
    except Exception:
        return None
    if "main_backtest" in payload:
        main = payload.get("main_backtest") or {}
        return {
            "return_pct": float(main.get("return", math.nan)),
            "max_drawdown_pct": float(main.get("max_dd", math.nan)),
            "summary_path": rel(path),
        }
    valid = payload.get("valid") or payload.get("test_proxy")
    if isinstance(valid, dict):
        return {
            "return_pct": float(valid.get("return", math.nan)),
            "max_drawdown_pct": float(valid.get("max_dd", math.nan)),
            "summary_path": rel(path),
        }
    if window in payload and isinstance(payload[window], dict):
        nested = payload[window]
        if "main_backtest" in nested:
            main = nested.get("main_backtest") or {}
            return {
                "return_pct": float(main.get("return", math.nan)),
                "max_drawdown_pct": float(main.get("max_dd", math.nan)),
                "summary_path": rel(path),
            }
    return None


freshness = load_json(ROOT / "dashboard_data/model_replay_freshness.json")
fresh_profiles = {
    str(item["profile"]): item
    for item in freshness.get("profiles", [])
    if isinstance(item, dict) and item.get("result") in {"current", "refreshed"}
}

profiles: dict[str, dict[str, Any]] = {}
for profile, item in fresh_profiles.items():
    profiles.setdefault(profile, {"profile": profile})
    profiles[profile].update({
        "status": item.get("status"),
        "actor": item.get("actor"),
        "profile_yaml": item.get("profile_yaml"),
        "ytd_daily_path": item.get("output"),
        "freshness_result": item.get("result"),
    })

for card_dir in sorted((ROOT / "runtime/model/card").iterdir()):
    if not card_dir.is_dir():
        continue
    profile = card_dir.name
    if any(profile.endswith(f"_{window}") for window in REQUIRED_WF):
        continue
    profiles.setdefault(profile, {"profile": profile})
    if "ytd_daily_path" not in profiles[profile]:
        preferred = card_dir / f"{profile}_soxl_daily.csv"
        if preferred.exists():
            profiles[profile]["ytd_daily_path"] = rel(preferred)
    for window in REQUIRED_WF:
        key = f"{window}_daily_path"
        if key not in profiles[profile]:
            candidates = sorted(card_dir.glob(f"*{window}*_soxl_daily.csv"))
            if candidates:
                profiles[profile][key] = rel(candidates[0])
        summary_key = f"{window}_summary_path"
        if summary_key not in profiles[profile]:
            summary_candidates = [
                card_dir / f"{profile}-{window}.json",
                card_dir / f"{profile}-{window}-seed42.json",
                card_dir / f"soxl_{profile}_{window}_summary.json",
                ROOT / "runtime/model/card" / f"{profile}_{window}" / f"soxl_{profile}_{window}_summary.json",
            ]
            for candidate in summary_candidates:
                if candidate.exists():
                    profiles[profile][summary_key] = rel(candidate)
                    break

source_rows = []
for profile, item in sorted(profiles.items()):
    row = {"profile": profile, **item}
    row["wf_daily_count"] = sum(1 for window in REQUIRED_WF if row.get(f"{window}_daily_path"))
    row["wf_summary_count"] = sum(1 for window in REQUIRED_WF if row.get(f"{window}_summary_path"))
    row["has_ytd_daily"] = bool(row.get("ytd_daily_path"))
    source_rows.append(row)
source_audit = pd.DataFrame(source_rows).fillna("")
source_audit = source_audit[(source_audit["has_ytd_daily"]) | (source_audit["wf_daily_count"] > 0) | (source_audit["wf_summary_count"] > 0)].copy()
source_audit.sort_values(["has_ytd_daily", "wf_daily_count", "wf_summary_count", "profile"], ascending=[False, False, False, True]).head(20)


,profile,status,actor,profile_yaml,ytd_daily_path,freshness_result,wf_2020_summary_path,wf_2022_summary_path,wf_2023_2024_summary_path,wf_daily_count,wf_summary_count,has_ytd_daily,wf_2020_daily_path,wf_2022_daily_path,wf_2023_2024_daily_path
39,balanced_rule_residual_imitation_v4,candidate,runtime/model/card/balanced_rule_residual_imit...,runtime/execution/card/balanced_rule_residual_...,runtime/model/card/balanced_rule_residual_imit...,current,runtime/model/card/balanced_rule_residual_imit...,runtime/model/card/balanced_rule_residual_imit...,runtime/model/card/balanced_rule_residual_imit...,3,3,True,runtime/model/card/balanced_rule_residual_imit...,runtime/model/card/balanced_rule_residual_imit...,runtime/model/card/balanced_rule_residual_imit...
45,balanced_rule_residual_mode_v1,,,,runtime/model/card/balanced_rule_residual_mode...,,runtime/model/card/balanced_rule_residual_mode...,runtime/model/card/balanced_rule_residual_mode...,runtime/model/card/balanced_rule_residual_mode...,3,3,True,runtime/model/card/balanced_rule_residual_mode...,runtime/model/card/balanced_rule_residual_mode...,runtime/model/card/balanced_rule_residual_mode...
75,drawdown_rebound_reward_train_v1,ablation,runtime/model/card/drawdown_rebound_reward_tra...,runtime/reward/card/drawdown_rebound_reward_tr...,runtime/model/card/drawdown_rebound_reward_tra...,current,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...,3,3,True,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...,runtime/model/card/drawdown_rebound_reward_tra...
79,mode_conditioned_rebound_reward_train_v2,ablation,runtime/model/card/mode_conditioned_rebound_re...,runtime/reward/card/mode_conditioned_rebound_r...,runtime/model/card/mode_conditioned_rebound_re...,current,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...,3,3,True,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...,runtime/model/card/mode_conditioned_rebound_re...
93,state_conditioned_drawdown_rebound_reward_trai...,ablation,runtime/model/card/state_conditioned_drawdown_...,runtime/reward/card/state_conditioned_drawdown...,runtime/model/card/state_conditioned_drawdown_...,current,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...,3,3,True,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...,runtime/model/card/state_conditioned_drawdown_...
0,balanced,archived,runtime/model/card/balanced/soxl_balanced_acto...,runtime/model/card/balanced/profile.yaml,runtime/model/card/balanced/balanced_soxl_dail...,current,runtime/model/card/balanced/balanced-wf_2020-s...,runtime/model/card/balanced/balanced-wf_2022-s...,runtime/model/card/balanced/balanced-wf_2023_2...,0,3,True,,,
55,balanced_rule_residual_v19,archived,runtime/model/card/balanced_rule_residual_v19/...,runtime/model/card/balanced_rule_residual_v19/...,runtime/model/card/balanced_rule_residual_v19/...,current,runtime/model/card/balanced_rule_residual_v19/...,runtime/model/card/balanced_rule_residual_v19/...,runtime/model/card/balanced_rule_residual_v19/...,0,3,True,,,
56,balanced_rule_residual_v19_lowcap,,,,runtime/model/card/balanced_rule_residual_v19_...,,runtime/model/card/balanced_rule_residual_v19_...,runtime/model/card/balanced_rule_residual_v19_...,runtime/model/card/balanced_rule_residual_v19_...,0,3,True,,,
59,balanced_rule_residual_v19_reg,candidate,runtime/model/card/balanced_rule_residual_v19_...,runtime/model/card/balanced_rule_residual_v19_...,runtime/model/card/balanced_rule_residual_v19_...,current,runtime/model/card/balanced_rule_residual_v19_...,runtime/model/card/balanced_rule_residual_v19_...,runtime/model/card/balanced_rule_resid

In [5]:

reference_by_window = {
    window: normalise_daily(REFERENCE_DAILY[(REFERENCE_DAILY["candidate"].eq("legacy_default")) & (REFERENCE_DAILY["window"].eq(window))])
    for window in [*REQUIRED_WF, "ytd_2026"]
}
risk_floor_by_window = {
    window: normalise_daily(REFERENCE_DAILY[(REFERENCE_DAILY["candidate"].eq("wf_pass_ytd18_15_15_02")) & (REFERENCE_DAILY["window"].eq(window))])
    for window in [*REQUIRED_WF, "ytd_2026"]
}

fold_rows: list[dict[str, Any]] = []
missing_rows: list[dict[str, Any]] = []

for _, profile_row in source_audit.iterrows():
    profile = str(profile_row["profile"])
    ytd_path = str(profile_row.get("ytd_daily_path") or "")
    if ytd_path:
        path = ROOT / ytd_path
        if path.exists():
            daily = pd.read_csv(path)
            ytd = window_slice(daily, YTD_START, YTD_END)
            metrics = metric_from_daily(ytd)
            align = alignment_metrics(ytd, reference_by_window["ytd_2026"])
            risk_align = alignment_metrics(ytd, risk_floor_by_window["ytd_2026"])
            fold_rows.append({
                "profile": profile,
                "window": "ytd_2026",
                "source_kind": "daily",
                "source_path": ytd_path,
                **metrics,
                **{f"baseline_{k}": v for k, v in align.items()},
                "risk_floor_position_mae": risk_align["position_mae"],
                "risk_floor_position_corr": risk_align["position_corr"],
            })
        else:
            missing_rows.append({"profile": profile, "window": "ytd_2026", "missing": "ytd_daily_path_not_found", "path": ytd_path})
    else:
        missing_rows.append({"profile": profile, "window": "ytd_2026", "missing": "missing_ytd_daily_path", "path": ""})

    for window in REQUIRED_WF:
        daily_path = str(profile_row.get(f"{window}_daily_path") or "")
        summary_path = str(profile_row.get(f"{window}_summary_path") or "")
        if daily_path and (ROOT / daily_path).exists():
            daily = pd.read_csv(ROOT / daily_path)
            metrics = metric_from_daily(normalise_daily(daily))
            align = alignment_metrics(daily, reference_by_window[window])
            risk_align = alignment_metrics(daily, risk_floor_by_window[window])
            fold_rows.append({
                "profile": profile,
                "window": window,
                "source_kind": "daily",
                "source_path": daily_path,
                **metrics,
                **{f"baseline_{k}": v for k, v in align.items()},
                "risk_floor_position_mae": risk_align["position_mae"],
                "risk_floor_position_corr": risk_align["position_corr"],
            })
        elif summary_path and (ROOT / summary_path).exists():
            metrics = metric_from_summary(ROOT / summary_path, window)
            if metrics is not None:
                source_path = metrics.pop("summary_path")
                fold_rows.append({
                    "profile": profile,
                    "window": window,
                    "source_kind": "summary",
                    "source_path": source_path,
                    **metrics,
                    "rows": 0,
                    "baseline_aligned_rows": 0,
                    "baseline_position_mae": math.nan,
                    "baseline_position_corr": math.nan,
                    "baseline_event_match_rate": math.nan,
                    "baseline_candidate_event_days": 0,
                    "baseline_reference_event_days": 0,
                    "baseline_action_mismatch_days": 0,
                    "risk_floor_position_mae": math.nan,
                    "risk_floor_position_corr": math.nan,
                })
            else:
                missing_rows.append({"profile": profile, "window": window, "missing": "summary_unparseable", "path": summary_path})
        else:
            missing_rows.append({"profile": profile, "window": window, "missing": "missing_wf_daily_or_summary", "path": daily_path or summary_path})

fold_metrics = pd.DataFrame(fold_rows)
missing_evidence = pd.DataFrame(missing_rows)
fold_metrics.head(20)


,profile,window,source_kind,source_path,return_pct,max_drawdown_pct,rows,baseline_aligned_rows,baseline_date_overlap_start,baseline_date_overlap_end,baseline_position_mae,baseline_position_corr,baseline_event_match_rate,baseline_candidate_event_days,baseline_reference_event_days,baseline_action_mismatch_days,risk_floor_position_mae,risk_floor_position_corr
0,balanced,ytd_2026,daily,runtime/model/card/balanced/balanced_soxl_dail...,146.723139,29.698281,109,109,2026-01-02,2026-06-09,0.477058,-0.662218,0.724771,27,3,30,0.332701,-0.314723
1,balanced,wf_2020,summary,runtime/model/card/balanced/balanced-wf_2020-s...,60.265174,27.156030,0,0,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN
2,balanced,wf_2022,summary,runtime/model/card/balanced/balanced-wf_2022-s...,34.460804,24.666145,0,0,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN
3,balanced,wf_2023_2024,summary,runtime/model/card/balanced/balanced-wf_2023_2...,47.954413,28.635516,0,0,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN
4,balanced_macro_v1,ytd_2026,daily,runtime/model/card/balanced_macro_v1/balanced_...,182.862242,29.901822,109,109,2026-01-02,2026-06-09,0.506309,-0.654480,0.752294,24,3,27,0.361952,-0.311046
5,balanced_rule_residual_imitation_v4,ytd_2026,daily,runtime/model/card/balanced_rule_residual_imit...,145.031235,25.999446,109,109,2026-01-02,2026-06-09,0.440678,-0.671295,0.862385,12,3,15,0.296321,-0.319037
6,balanced_rule_residual_imitation_v4,wf_2020,daily,runtime/model/card/balanced_rule_residual_imit...,102.528303,20.757988,253,253,2020-01-02,2020-12-31,0.480603,0.200761,0.810277,44,4,48,0.355182,-0.287024
7,balanced_rule_residual_imitation_v4,wf_2022,daily,runtime/model/card/balanced_rule_residual_imit...,-28.268295,37.705459,251,251,2022-01-03,2022-12-30,0.650754,-0.161063,0.836653,33,8,41,0.169589,-0.025678
8,balanced_rule_residual_imitation_v4,wf_2023_2024,daily,runtime/model/card/balanced_rule_residual_imit...,-16.298996,34.464267,252,252,2024-01-02,2024-12-31,0.381709,-0.799875,0.825397,40,4,44,0.317415,-0.168280
9,balanced_rule_residual_mode_v1,ytd_2026,daily,runtime/model/card/balanced_rule_residual_mode...,7.296552,21.903644,61,61,2026-01-02,2026-03-31,0.009830,NaN,0.737705,15,1,16,0.004915,NaN


In [6]:

def safe_mean(values: pd.Series) -> float:
    vals = pd.to_numeric(values, errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else math.nan

summary_rows: list[dict[str, Any]] = []
for profile, group in fold_metrics.groupby("profile", sort=False):
    ytd = group[group["window"].eq("ytd_2026")]
    wf = group[group["window"].isin(REQUIRED_WF)]
    ytd_return = float(ytd.iloc[0]["return_pct"]) if not ytd.empty else math.nan
    ytd_dd = float(ytd.iloc[0]["max_drawdown_pct"]) if not ytd.empty else math.nan
    wf_present = sorted(wf["window"].astype(str).unique().tolist())
    missing_wf = sorted(set(REQUIRED_WF) - set(wf_present))
    worst_wf_dd = float(pd.to_numeric(wf["max_drawdown_pct"], errors="coerce").max()) if not wf.empty else math.nan
    min_wf_return = float(pd.to_numeric(wf["return_pct"], errors="coerce").min()) if not wf.empty else math.nan
    mean_wf_return = safe_mean(wf["return_pct"]) if not wf.empty else math.nan
    mean_pos_mae = safe_mean(group["baseline_position_mae"])
    mean_event_match = safe_mean(group["baseline_event_match_rate"])
    mean_risk_mae = safe_mean(group["risk_floor_position_mae"])
    complete_wf = not missing_wf
    wf_drawdown_pass = bool(complete_wf and pd.notna(worst_wf_dd) and worst_wf_dd < MAX_WF_DD)
    soxl_return_pass = bool(pd.notna(ytd_return) and ytd_return + 1e-6 >= REQ_SOXL_RETURN)
    best_return_pass = bool(pd.notna(ytd_return) and ytd_return + 1e-6 >= REQ_BEST_RETURN)
    retention_gate_pass = bool(wf_drawdown_pass and soxl_return_pass and best_return_pass)
    failures: list[str] = []
    failures.extend(f"missing_wf_window:{item}" for item in missing_wf)
    if complete_wf and not wf_drawdown_pass:
        failures.append("worst_wf_max_drawdown_ge_20")
    if not soxl_return_pass:
        failures.append("ytd_return_below_soxlbacktest_reference")
    if not best_return_pass:
        failures.append("ytd_return_below_current_best_alignment")
    # Positive score favors return and complete evidence, but heavily penalizes drawdown-gate failures.
    score = 0.0
    score += 0.55 * (ytd_return if pd.notna(ytd_return) else -100.0)
    score += 0.25 * (mean_wf_return if pd.notna(mean_wf_return) else -100.0)
    score -= 2.0 * max(0.0, (worst_wf_dd if pd.notna(worst_wf_dd) else 100.0) - MAX_WF_DD)
    score -= 25.0 * len(missing_wf)
    score -= 10.0 * (mean_pos_mae if pd.notna(mean_pos_mae) else 0.5)
    summary_rows.append({
        "profile": profile,
        "status": str(source_audit.set_index("profile").to_dict("index").get(profile, {}).get("status", "")),
        "has_ytd_daily": bool(not ytd.empty),
        "wf_windows_present": ",".join(wf_present),
        "missing_wf_windows": ";".join(missing_wf),
        "ytd_return_pct": round(ytd_return, 6) if pd.notna(ytd_return) else math.nan,
        "ytd_max_drawdown_pct": round(ytd_dd, 6) if pd.notna(ytd_dd) else math.nan,
        "worst_wf_max_drawdown_pct": round(worst_wf_dd, 6) if pd.notna(worst_wf_dd) else math.nan,
        "min_wf_return_pct": round(min_wf_return, 6) if pd.notna(min_wf_return) else math.nan,
        "mean_wf_return_pct": round(mean_wf_return, 6) if pd.notna(mean_wf_return) else math.nan,
        "mean_baseline_position_mae": round(mean_pos_mae, 6) if pd.notna(mean_pos_mae) else math.nan,
        "mean_baseline_event_match_rate": round(mean_event_match, 6) if pd.notna(mean_event_match) else math.nan,
        "mean_risk_floor_position_mae": round(mean_risk_mae, 6) if pd.notna(mean_risk_mae) else math.nan,
        "complete_wf": complete_wf,
        "wf_drawdown_pass": wf_drawdown_pass,
        "ytd_soxlbacktest_return_pass": soxl_return_pass,
        "ytd_current_best_alignment_pass": best_return_pass,
        "retention_gate_pass": retention_gate_pass,
        "selection_score": round(score, 6),
        "failures": ";".join(failures),
    })

candidate_scores = pd.DataFrame(summary_rows).sort_values(
    ["retention_gate_pass", "complete_wf", "wf_drawdown_pass", "selection_score"],
    ascending=[False, False, False, False],
).reset_index(drop=True)
candidate_scores.head(30)


,profile,status,has_ytd_daily,wf_windows_present,missing_wf_windows,ytd_return_pct,ytd_max_drawdown_pct,worst_wf_max_drawdown_pct,min_wf_return_pct,mean_wf_return_pct,mean_baseline_position_mae,mean_baseline_event_match_rate,mean_risk_floor_position_mae,complete_wf,wf_drawdown_pass,ytd_soxlbacktest_return_pass,ytd_current_best_alignment_pass,retention_gate_pass,selection_score,failures
0,state_conditioned_drawdown_rebound_reward_trai...,ablation,True,"wf_2020,wf_2022,wf_2023_2024",,196.370488,31.390931,29.147985,-23.275008,10.113392,0.501859,0.838304,0.218346,True,False,True,False,False,87.217557,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
1,drawdown_rebound_reward_train_v1,ablation,True,"wf_2020,wf_2022,wf_2023_2024",,161.708312,27.310377,20.934520,-7.000281,14.549028,0.546827,0.856260,0.167268,True,False,True,False,False,85.239518,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
2,balanced,archived,True,"wf_2020,wf_2022,wf_2023_2024",,146.723139,29.698281,28.635516,34.460804,47.560130,0.477058,0.724771,0.332701,True,False,True,False,False,70.546149,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
3,mode_conditioned_rebound_reward_train_v2,ablation,True,"wf_2020,wf_2022,wf_2023_2024",,165.609065,27.278641,30.489188,-16.085903,7.097381,0.534013,0.835046,0.226825,True,False,True,False,False,66.540825,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
4,balanced_rule_residual_v19,archived,True,"wf_2020,wf_2022,wf_2023_2024",,173.299227,24.399161,46.653957,-41.932325,31.505807,0.458983,0.862385,0.314626,True,False,True,False,False,45.293281,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
5,balanced_rule_residual_imitation_v4,candidate,True,"wf_2020,wf_2022,wf_2023_2024",,145.031235,25.999446,37.705459,-28.268295,19.320337,0.488436,0.833678,0.284627,True,False,True,False,False,44.301985,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
6,balanced_rule_residual_v19_reg_light,candidate,True,"wf_2020,wf_2022,wf_2023_2024",,164.205635,25.990693,51.596776,-44.718094,32.697421,0.452026,0.862385,0.307669,True,False,True,False,False,30.773643,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
7,balanced_rule_residual_v19_reg,candidate,True,"wf_2020,wf_2022,wf_2023_2024",,141.992184,25.513909,46.171637,-40.495062,27.658455,0.439189,0.862385,0.294832,True,False,True,False,False,28.275148,worst_wf_max_drawdown_ge_20;ytd_return_below_c...
8,balanced_rule_residual_v22,canonical,True,"wf_2020,wf_2022,wf_2023_2024",,116.882644,26.278174,54.665610,-48.403668,46.471605,0.438603,0.880734,0.294247,True,False,False,False,False,2.186101,worst_wf_max_drawdown_ge_20;ytd_return_below_s...
9,balanced_rule_residual_v19_small,,True,"wf_2020,wf_2022,wf_2023_2024",,0.505721,19.884946,30.655663,49.724869,59.810346,0.009830,0.983607,0.004915,True,False,False,False,False,-6.178889,worst_wf_max_drawdown_ge_20;ytd_return_below_s...


In [7]:

passing = candidate_scores[candidate_scores["retention_gate_pass"]]
complete = candidate_scores[candidate_scores["complete_wf"]]
wf_dd_pass = candidate_scores[candidate_scores["wf_drawdown_pass"]]
high_ytd = candidate_scores[candidate_scores["ytd_soxlbacktest_return_pass"]]

selected_next_training_source = str(candidate_scores.iloc[0]["profile"]) if not candidate_scores.empty else ""
closest_complete_profile = str(complete.iloc[0]["profile"]) if not complete.empty else ""
best_ytd_profile = str(candidate_scores.sort_values("ytd_return_pct", ascending=False).iloc[0]["profile"]) if not candidate_scores.empty else ""
best_wf_dd_profile = str(complete.sort_values("worst_wf_max_drawdown_pct", ascending=True).iloc[0]["profile"]) if not complete.empty else ""

selection = {
    "experiment": "soxlbacktest_reference_model_score_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "no_existing_model_passes_retention_gate_use_reference_scoring_for_next_training",
    "retention_gate_pass_candidates": passing["profile"].tolist(),
    "selected_next_training_source": selected_next_training_source,
    "closest_complete_profile": closest_complete_profile,
    "best_ytd_profile": best_ytd_profile,
    "best_wf_drawdown_profile": best_wf_dd_profile,
    "thresholds": thresholds,
    "top_candidate": candidate_scores.iloc[0].to_dict() if not candidate_scores.empty else {},
    "soxlbacktest_reference": {
        "buy_sell_reference": "legacy_default",
        "risk_floor_reference": BASELINE_SUMMARY["selected_risk_floor"],
        "source_card": "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1",
    },
    "sidecars": {name: rel(path) for name, path in ARTIFACTS.items()},
    "next_action": "train_or_select_from_the_highest_scored_complete_wf_family_but_keep_promotion_blocked_until_worst_wf_dd_lt_20_and_ytd_return_gates_pass",
}
selection


{'experiment': 'soxlbacktest_reference_model_score_v1',
 'created_at': '2026-06-13T11:05:52.513204+00:00',
 'decision': 'no_existing_model_passes_retention_gate_use_reference_scoring_for_next_training',
 'retention_gate_pass_candidates': [],
 'selected_next_training_source': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'closest_complete_profile': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'best_ytd_profile': 'smoke',
 'best_wf_drawdown_profile': 'drawdown_rebound_reward_train_v1',
 'thresholds': {'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
  'max_walkforward_drawdown_pct': 20.0,
  'required_soxlbacktest_return_pct': 132.077978,
  'required_best_return_pct': 213.143462,
  'ytd_start': '2026-01-02',
  'ytd_end': '2026-06-09'},
 'top_candidate': {'profile': 'state_conditioned_drawdown_rebound_reward_train_v1',
  'status': 'ablation',
  'has_ytd_daily': True,
  'wf_windows_present': 'wf_2020,wf_2022,wf_2023_2024',
  'missing_wf_windows': '',
  'ytd_

In [8]:

source_audit.to_csv(ARTIFACTS["profile_source_audit"], index=False)
fold_metrics.to_csv(ARTIFACTS["fold_alignment_metrics"], index=False)
missing_evidence.to_csv(ARTIFACTS["missing_evidence"], index=False)
candidate_scores.to_csv(ARTIFACTS["candidate_score_summary"], index=False)
ARTIFACTS["selection"].write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary = {
    "experiment": "soxlbacktest_reference_model_score_v1",
    "created_at": selection["created_at"],
    "decision": selection["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "retention_gate_pass_candidates": selection["retention_gate_pass_candidates"],
    "profiles_scored": int(len(candidate_scores)),
    "complete_wf_profiles": int(candidate_scores["complete_wf"].sum()) if not candidate_scores.empty else 0,
    "wf_drawdown_pass_profiles": wf_dd_pass["profile"].tolist(),
    "ytd_soxlbacktest_return_pass_profiles": high_ytd["profile"].tolist(),
    "selected_next_training_source": selection["selected_next_training_source"],
    "closest_complete_profile": selection["closest_complete_profile"],
    "best_ytd_profile": selection["best_ytd_profile"],
    "best_wf_drawdown_profile": selection["best_wf_drawdown_profile"],
    "thresholds": thresholds,
    "sidecars": {name: rel(path) for name, path in ARTIFACTS.items()},
}
ARTIFACTS["summary"].write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

conclusion_lines = [
    "# soxlbacktest_reference_model_score_v1",
    "",
    "- Decision: `no_existing_model_passes_retention_gate_use_reference_scoring_for_next_training`.",
    f"- Profiles scored: `{summary['profiles_scored']}`; complete WF profiles: `{summary['complete_wf_profiles']}`.",
    f"- Retention-gate pass candidates: `{', '.join(summary['retention_gate_pass_candidates']) if summary['retention_gate_pass_candidates'] else 'none'}`.",
    f"- Selected next-training source by score: `{summary['selected_next_training_source']}`.",
    f"- Best YTD profile: `{summary['best_ytd_profile']}`; best WF drawdown profile among complete-WF candidates: `{summary['best_wf_drawdown_profile']}`.",
    "- SOXLBacktest `legacy_default` remains the buy/sell reference; `wf_pass_ytd18_15_15_02` remains a low-drawdown but low-return risk floor.",
    "- Promotion remains blocked until one candidate has all required WF windows, worst WF MaxDD <20%, YTD return >= SOXLBacktest reference, and YTD return >= current-best alignment floor.",
    "",
    "## Sidecars",
]
conclusion_lines.extend(f"- `{rel(path)}`" for path in ARTIFACTS.values())
ARTIFACTS["conclusion"].write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")
summary


{'experiment': 'soxlbacktest_reference_model_score_v1',
 'created_at': '2026-06-13T11:05:52.513204+00:00',
 'decision': 'no_existing_model_passes_retention_gate_use_reference_scoring_for_next_training',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'retention_gate_pass_candidates': [],
 'profiles_scored': 21,
 'complete_wf_profiles': 12,
 'wf_drawdown_pass_profiles': [],
 'ytd_soxlbacktest_return_pass_profiles': ['state_conditioned_drawdown_rebound_reward_train_v1',
  'drawdown_rebound_reward_train_v1',
  'balanced',
  'mode_conditioned_rebound_reward_train_v2',
  'balanced_rule_residual_v19',
  'balanced_rule_residual_imitation_v4',
  'balanced_rule_residual_v19_reg_light',
  'balanced_rule_residual_v19_reg',
  'smoke',
  'balanced_macro_v1',
  'balanced_rule_residual_v19_reg_v22_retest_v1'],
 'selected_next_training_source': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'closest_complete_profile': 'state_conditioned_drawdown_rebound_reward_train_v1',
 'bes

## Conclusion

This notebook keeps the final-model gate closed. It converts the qlib-aligned SOXLBacktest reference into a model-selection scorer and records which existing candidates have missing evidence, excessive walk-forward drawdown, or insufficient YTD return. The result should guide the next training loop rather than promote any existing actor.
